# 🧪 Taller Práctico: Transformers, BERT y RoBERTa

Este notebook acompaña una clase práctica de PLN sobre modelos de lenguaje basados en Transformers. Usaremos modelos preentrenados como BERT y RoBERTa para realizar tareas de análisis de sentimientos, clasificación de texto y análisis de representaciones.

## ✅ Parte 1: Uso de `pipeline` para tareas básicas de NLP

In [3]:
from transformers import pipeline

# Clasificación de sentimiento
task = pipeline("sentiment-analysis")
text = "BERT es un modelo muy poderoso para NLP."
result = task(text)
print(result)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


[{'label': 'NEGATIVE', 'score': 0.9747099280357361}]


In [13]:
# Load the fine-tuned BERT-Emotion model
sentiment_analysis = pipeline("text-classification", model="boltuix/bert-emotion")

# Define label-to-emoji mapping
label_to_emoji = {
    "Sadness": "😢",
    "Anger": "😠",
    "Love": "❤️",
    "Surprise": "😲",
    "Fear": "😱",
    "Happiness": "😄",
    "Neutral": "😐",
    "Disgust": "🤢",
    "Shame": "🙈",
    "Guilt": "😔",
    "Confusion": "😕",
    "Desire": "🔥",
    "Sarcasm": "😏"
}

# Input text
text = "Meh"

# Analyze emotion
result = sentiment_analysis(text)[0]
label = result["label"].capitalize()
emoji = label_to_emoji.get(label, "❓")

# Output
print(f"Text: {text}")
print(f"Predicted Emotion: {label} {emoji}")
print(f"Confidence: {result['score']:.2%}")


Device set to use cpu


Text: Meh
Predicted Emotion: Anger 😠
Confidence: 32.49%


## ✅ Parte 2: Tokenización y extracción de embeddings

In [15]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

text = "Transformers are powerful models for NLP."
inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

print("Shape del embedding:", outputs.last_hidden_state.shape)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Shape del embedding: torch.Size([1, 10, 768])


## ✅ Parte 3: Clasificación de texto con RoBERTa y dataset AG News

In [16]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
from sklearn.metrics import accuracy_score

In [20]:
# Cargar dataset AG News
import pandas as pd
from datasets import Dataset, DatasetDict

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "test": Dataset.from_pandas(test_df)
})

In [36]:
dataset = dataset.rename_column("Class Index", "labels")
# Obtener los nombres únicos de clases (por ejemplo: 'World', 'Sports', etc.)
label_list = list(set(dataset["train"]["labels"]))
label_list.sort()  # Para mantener un orden estable

# Mapear string → int
label2id = {label: i for i, label in enumerate(label_list)}

# Aplicar el mapeo
def encode_labels(example):
    example["labels"] = label2id[example["labels"]]
    return example

dataset = dataset.map(encode_labels)
def join_text(example):
    example["text"] = example["Title"] + " " + example["Description"]
    return example

dataset = dataset.map(join_text)


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [22]:
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [37]:
def preprocess(example):
    return tokenizer(example["text"], truncation=True)

tokenized_ds = dataset.map(preprocess, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [38]:
args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=10,
    logging_dir='./logs',  # requerido en versiones antiguas
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"].shuffle(seed=42).select(range(1000)),
    eval_dataset=tokenized_ds["test"].select(range(200)),
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
eval_result = trainer.evaluate()
print(eval_result)

<ipython-input-38-2831514812>:11: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
10,1.343400
20,1.067700
30,0.611700
40,0.312900
50,0.436300
60,0.335900


{'eval_loss': 0.3721379041671753, 'eval_accuracy': 0.88, 'eval_runtime': 71.4203, 'eval_samples_per_second': 2.8, 'eval_steps_per_second': 0.182, 'epoch': 1.0}


## ✅ Parte 4: Adaptación de RoBERTa en español para clasificación

En esta sección usaremos el modelo `PlanTL-GOB-ES/roberta-base-bne` (RoBERTa entrenado en corpus en español) y un dataset en español para clasificar sentimientos de tweets.

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification

# Dataset de ejemplo en español
es_dataset = load_dataset("pysentimiento/tweets-sentiment", split="train[:2000]")

model_name = "PlanTL-GOB-ES/roberta-base-bne"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

def preprocess_es(example):
    return tokenizer(example['text'], truncation=True)

encoded_dataset = es_dataset.map(preprocess_es, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Métricas
import numpy as np

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="no",
    per_device_train_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()